# Mathématiques appliquées à l'Intelligence Artificielle 
## 📝 Jour 2 : Optimisation
## 📚 Complément du cours 

2e année Bachelor Informatique

---

> © 2025 Mohamed EL AFRIT
> Ce contenu est distribué sous licence [Creative Commons BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/deed.fr)
> [www.mohamedelafrit.com](https://www.mohamedelafrit.com)


## 🎯 Objectifs du Jour 2
- Comprendre la **fonction de coût** (MSE) comme mesure d'erreur
- Saisir l'intuition de la **dérivée** (pente d'une fonction)
- Implémenter la **descente de gradient** pour optimiser les poids
- Comprendre l'impact du **taux d'apprentissage** $\alpha$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# DATASET FIL ROUGE — 30 développeurs
# ============================================================
# Colonnes : experience, nb_langages, niveau_etudes,
#            taille_entreprise, remote, salaire
noms_colonnes = ["experience", "nb_langages", "niveau_etudes",
                 "taille_entreprise", "remote", "salaire"]

data = np.array([
    [ 0, 2, 3,  150,  80, 28.0],  # 0  junior
    [ 1, 1, 2,   50, 100, 26.5],  # 1  autodidacte
    [ 1, 3, 4,  800,  40, 35.0],  # 2  sortie ecole
    [ 2, 2, 3,  200,  60, 32.0],  # 3
    [ 2, 4, 4, 3000,  20, 38.0],  # 4
    [ 0, 1, 1,   30,  50, 25.0],  # 5  debutant
    [ 3, 3, 3,  100,  80, 34.5],  # 6
    [ 1, 5, 4,   15, 100, 42.0],  # 7  junior polyglotte
    [ 4, 3, 3,  500,  60, 38.5],  # 8
    [ 5, 4, 4, 1200,  40, 44.0],  # 9
    [ 5, 3, 3,  300,  80, 41.0],  # 10
    [ 6, 5, 4, 2000,  30, 48.0],  # 11
    [ 6, 4, 3,  150,  90, 43.5],  # 12
    [ 7, 5, 4,  800,  50, 50.0],  # 13
    [ 7, 3, 2,   80,  70, 39.0],  # 14 exp mais Bac+2
    [ 4, 6, 5,  400, 100, 47.0],  # 15
    [ 8, 4, 4, 1500,  40, 51.0],  # 16
    [ 5, 2, 3, 4000,  10, 40.0],  # 17
    [ 9, 5, 4,  600,  60, 52.0],  # 18
    [10, 6, 4, 2500,  30, 55.0],  # 19
    [10, 4, 3,  200,  80, 48.5],  # 20
    [11, 5, 5, 1000,  50, 58.0],  # 21
    [12, 7, 4, 3500,  20, 60.0],  # 22
    [ 9, 3, 3,   60,  90, 42.0],  # 23 senior sous-paye
    [13, 6, 4,  800,  70, 62.0],  # 24
    [11, 4, 4, 5000,  10, 54.0],  # 25
    [15, 8, 4, 1200,  50, 65.0],  # 26
    [17, 6, 5, 2000,  40, 68.0],  # 27
    [20, 7, 4,  500,  60, 72.0],  # 28
    [18, 5, 3,  100,  80, 58.0],  # 29
])

experience = data[:, 0]
nb_langages = data[:, 1]
salaire = data[:, 5]
salaire_eleve = (salaire > 45).astype(int)
n = len(data)

print(f"Dataset charge : {n} developpeurs, {data.shape[1]} colonnes")
print(f"Classe 0 (<=45k) : {np.sum(salaire_eleve==0)}, Classe 1 (>45k) : {np.sum(salaire_eleve==1)}")


---
## 1. Fonction de coût (MSE)

> **Définition** : La fonction de coût mesure l'erreur du modèle :
> $$J(w) = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$
>
> **Objectif** : trouver $w^* = \arg\min_w J(w)$, le « fond de la vallée ».


In [ ]:
# Visualiser J(w1) pour différentes valeurs de w1 (w0 fixé à 31)
x = experience; y = salaire
w1_range = np.linspace(-5, 10, 200)
mse_values = [np.mean((y - (w1*x + 31))**2) for w1 in w1_range]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(w1_range, mse_values, 'b-', lw=2)
ax.set_xlabel("w₁"); ax.set_ylabel("J(w₁)")
ax.set_title("Fonction de coût — forme de parabole (convexe)")
w1_opt = w1_range[np.argmin(mse_values)]
ax.axvline(w1_opt, color='red', ls='--', label=f'Minimum ≈ {w1_opt:.1f}')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


---
## 2. La dérivée (intuition)

> La **dérivée** $f'(x)$ donne la **pente** de la fonction au point $x$.
>
> | Fonction | Dérivée |
> |----------|---------|
> | $ax^2$ | $2ax$ |
> | $(x-a)^2$ | $2(x-a)$ |
> | $ax + b$ | $a$ |
>
> Au **minimum**, la pente est nulle : $f'(x^*) = 0$.


In [ ]:
# Exemple : f(w) = (w - 3)²
def f(w): return (w - 3)**2
def df(w): return 2*(w - 3)

w_range = np.linspace(-1, 7, 100)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(w_range, f(w_range), 'b-', lw=2); ax1.set_title("f(w) = (w-3)²")
ax1.axvline(3, color='red', ls='--', label='Minimum w*=3'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(w_range, df(w_range), 'g-', lw=2); ax2.set_title("f'(w) = 2(w-3)")
ax2.axhline(0, color='gray', ls=':'); ax2.axvline(3, color='red', ls='--'); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


---
## 3. Descente de gradient

> **Algorithme** : partir d'un point initial, avancer dans la direction opposée au gradient :
> $$w \leftarrow w - \alpha \cdot \frac{\partial J}{\partial w}$$
>
> **Intuition** : descendre une montagne les yeux bandés en suivant la pente sous ses pieds.


In [ ]:
# Descente de gradient sur f(w) = (w-3)²
w = 0.0          # point de départ
alpha = 0.1      # taux d'apprentissage
n_iter = 20

print(f"{'Iter':>4} | {'w':>8} | {'f(w)':>8} | {'f\'(w)':>8}")
print("-" * 40)
for i in range(n_iter):
    grad = 2 * (w - 3)       # dérivée
    w = w - alpha * grad      # mise à jour
    if i < 5 or i == n_iter-1:
        print(f"{i+1:4d} | {w:8.4f} | {f(w):8.4f} | {grad:8.4f}")

print(f"\nRésultat : w = {w:.4f} (optimal : 3.0)")


---
## 4. Taux d'apprentissage $\alpha$


In [ ]:
# Comparer 4 valeurs de alpha
alphas = [0.001, 0.05, 0.1, 0.9]
colors = ['#e74c3c', '#f39c12', '#2ecc71', '#9b59b6']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
w_range = np.linspace(-3, 8, 200)

for alpha, color in zip(alphas, colors):
    w = -2.0; hist_J = [f(w)]
    for _ in range(50):
        w = w - alpha * 2*(w-3); hist_J.append(f(w))
    ax1.plot(hist_J[:20], '-', color=color, lw=2, label=f'α={alpha}')
    ax2.plot(range(len(hist_J)), hist_J, '-', color=color, lw=2, label=f'α={alpha}')

ax1.set_title("20 premières itérations"); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.set_title("Convergence (50 iter)"); ax2.set_xlabel("Itération"); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print("α trop petit → lent | α bon → converge vite | α trop grand → oscille/diverge")


---
## 5. Application : régression sur le dataset salaire


In [ ]:
# Descente de gradient pour la régression linéaire
x = experience; y = salaire; n = len(y)
w1, w0 = 0.0, 0.0
alpha = 0.001
n_iter = 5000
hist_mse = []

for it in range(n_iter):
    y_pred = w1 * x + w0
    errors = y - y_pred
    dw1 = -2/n * np.sum(x * errors)
    dw0 = -2/n * np.sum(errors)
    w1 -= alpha * dw1
    w0 -= alpha * dw0
    hist_mse.append(np.mean(errors**2))

print(f"Gradient descent : ŷ = {w1:.4f}x + {w0:.4f}")
print(f"MSE = {hist_mse[-1]:.2f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(hist_mse, 'b-', lw=1.5); ax1.set_xlabel("Itération"); ax1.set_ylabel("MSE")
ax1.set_title("Convergence"); ax1.grid(True, alpha=0.3)

ax2.scatter(x, y, c='blue', s=40, label='Données')
xl = np.linspace(0, 21, 100)
ax2.plot(xl, w1*xl+w0, 'r-', lw=2, label=f'GD: ŷ={w1:.2f}x+{w0:.2f}')
ax2.set_xlabel("Expérience"); ax2.set_ylabel("Salaire (k€)")
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


---
## ✅ Résumé du Jour 2

| Concept | Formule | Intuition |
|---------|---------|-----------|
| Coût (MSE) | $J = \frac{1}{n}\sum(y_i-\hat{y}_i)^2$ | Mesure de l'erreur |
| Dérivée | $f'(x) = \lim_{h→0}\frac{f(x+h)-f(x)}{h}$ | Pente locale |
| Gradient descent | $w ← w - α∇J$ | Descendre la montagne |
| Taux α | $α ∈ (0, 1)$ | Taille des pas |

> **Jour 3** : au lieu de prédire un nombre, on va **classifier** (salaire > 45k€ ?)
